# PySpark concepts, end to end

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/pyspark/pyspark-concepts-end-to-end.ipynb)

You own a daily orders pipeline. Day one includes malformed records and repeated order updates. The next delivery includes a late refund, a stale update, and new orders. Finance expects the revenue table to reflect each event once, even when the scheduler retries the delivery.

Follow the data from raw files to typed records and quarantine, then to one current row per order and a daily revenue report. Inspect the expected counts at each boundary. Replay the second delivery and verify that the logical results match and that an untouched silver partition keeps its files.

This lab uses a single local writer and assumes that an order's business date stays fixed. The platform notes explain what needs to change before this becomes a production deployment.

In [ ]:
# Setup: installs this notebook's packages when they are missing (e.g. in Google Colab).
import importlib.util
import subprocess
import sys

PACKAGES = {"pyspark": "pyspark>=4.0,<4.1"}


def _missing(module: str, spec: str) -> bool:
    """True when the module is absent or its installed version does not satisfy spec."""
    try:
        if importlib.util.find_spec(module) is None:
            return True
    except ModuleNotFoundError:
        return True
    try:
        from importlib.metadata import PackageNotFoundError, version

        from packaging.requirements import Requirement
    except ImportError:
        return False
    requirement = Requirement(spec)
    try:
        return not requirement.specifier.contains(version(requirement.name), prereleases=True)
    except PackageNotFoundError:
        return False


missing = [spec for module, spec in PACKAGES.items() if _missing(module, spec)]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
print("ready:", ", ".join(PACKAGES))

## The pipeline at a glance

| Stage | What it does | Module that teaches it |
| --- | --- | --- |
| 0. Session and landing zone | Local session, AQE on, raw files written to a temp folder | [PySpark 1](pyspark-01-foundations.ipynb) |
| 1. Bronze | Explicit schemas, `PERMISSIVE` reads, a quarantine for bad rows | [PySpark 2](pyspark-02-reading-sql-and-cleaning.ipynb), [PySpark 5](pyspark-05-writing-pipelines-and-quality.ipynb) |
| 2. Silver | Trim and normalise, latest-row dedupe with a window, `explode_outer` on nested events | [PySpark 2](pyspark-02-reading-sql-and-cleaning.ipynb), [PySpark 4](pyspark-04-windows-and-nested-data.ipynb) |
| 3. Gold | Broadcast join to the customer dimension, daily revenue, per-customer ranking | [PySpark 3](pyspark-03-joins-and-aggregations.ipynb), [PySpark 4](pyspark-04-windows-and-nested-data.ipynb) |
| 4. Incremental re-run | Dynamic partition overwrite of only the touched dates, run twice | [PySpark 5](pyspark-05-writing-pipelines-and-quality.ipynb) |
| 5. Data quality | Uniqueness, nulls, ranges, referential integrity, reconciliation | [PySpark 5](pyspark-05-writing-pipelines-and-quality.ipynb) |
| 6. Performance check | Broadcast vs sort-merge plan, partition pruning, one file per partition | [PySpark 7](pyspark-07-monitoring-and-performance.ipynb) |

Every stage ends with `assert`s, so "Run all" either proves the pipeline works or stops at the
first broken stage. Streaming is out of scope here: the same bronze/silver/gold shape on a stream
is covered in [PySpark 6](pyspark-06-structured-streaming.ipynb).

## Stage 0 - Session and landing zone

Uses: building a local `SparkSession` and reading its config ([PySpark 1](pyspark-01-foundations.ipynb)). Two settings matter for the
pipeline: `partitionOverwriteMode=dynamic`, so an overwrite replaces only the partitions present
in the new data, and a small `shuffle.partitions` because the data is tiny.

In [ ]:
import os
import sys

from pyspark.sql import SparkSession

# Python workers must use this notebook's interpreter.
os.environ["PYSPARK_PYTHON"] = sys.executable

spark = (
    SparkSession.builder
    .appName("de-pyspark-capstone")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.sources.partitionOverwriteMode", "dynamic")
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

print("Spark version :", spark.version)
print("Master        :", spark.sparkContext.master)

In [ ]:
assert spark.sparkContext.master.startswith("local"), spark.sparkContext.master
assert spark.conf.get("spark.sql.sources.partitionOverwriteMode") == "dynamic"
assert spark.conf.get("spark.sql.adaptive.enabled") == "true"
print("stage 0 passed: local session with dynamic partition overwrite and AQE")

### Land the day-1 raw files

The source systems drop CSV and JSON-lines files into a landing folder. The day-1 files are
deliberately messy, the way real extracts are:

- **orders**: a status with stray case and whitespace, an order that was updated later the same
  day (two versions of order 3), an exact re-delivered duplicate (order 2), an unparseable amount,
  an unparseable timestamp, a missing key, a negative amount, and an order for a customer (`c9`)
  that is not in the customer extract.
- **customers**: untrimmed names and mixed-case country codes.
- **events**: a duplicated event, an empty and a missing `items` array, and one truncated JSON line.

In [ ]:
import tempfile

LAKE = tempfile.mkdtemp(prefix="de-pyspark-capstone-")
LANDING = os.path.join(LAKE, "landing")
os.makedirs(LANDING)

ORDERS_HEADER = "order_id,customer_id,amount,status,order_ts,updated_at"
RAW_FILES = {
    "orders_day1.csv": [
        ORDERS_HEADER,
        "1,c1,120.50,paid,2026-10-01 10:00:00,2026-10-01 10:00:00",
        "2,c2,80.00,PAID ,2026-10-01 11:30:00,2026-10-01 11:30:00",
        "3,c3,200.00,shipped,2026-10-02 09:15:00,2026-10-02 09:15:00",
        "3,c3,200.00,delivered,2026-10-02 09:15:00,2026-10-02 18:00:00",
        "4,c1,15.75,cancelled,2026-10-02 12:00:00,2026-10-02 12:00:00",
        "5,c9,60.00,paid,2026-10-03 08:45:00,2026-10-03 08:45:00",
        "6,c2,abc,paid,2026-10-03 16:20:00,2026-10-03 16:20:00",
        "7,c4,45.00,paid,not-a-date,not-a-date",
        ",c4,10.00,paid,2026-10-03 17:00:00,2026-10-03 17:00:00",
        "8,c4,-5.00,paid,2026-10-03 18:00:00,2026-10-03 18:00:00",
        "2,c2,80.00,PAID ,2026-10-01 11:30:00,2026-10-01 11:30:00",
    ],
    "customers.csv": [
        "customer_id,name,country",
        "c1, Ada ,us",
        "c2,Grace,GB",
        "c3,Linus,fi",
        "c4,Margaret,US",
    ],
    "events_day1.jsonl": [
        '{"event_id": "e1", "customer_id": "c1", "event_type": "view", "event_ts": "2026-10-01 09:55:00", "items": [{"sku": "A", "qty": 1}]}',
        '{"event_id": "e2", "customer_id": "c1", "event_type": "cart", "event_ts": "2026-10-01 09:58:00", "items": [{"sku": "A", "qty": 1}, {"sku": "B", "qty": 2}]}',
        '{"event_id": "e3", "customer_id": "c2", "event_type": "view", "event_ts": "2026-10-01 11:00:00", "items": []}',
        '{"event_id": "e4", "customer_id": "c3", "event_type": "view", "event_ts": "2026-10-02 09:00:00"}',
        '{"event_id": "e4", "customer_id": "c3", "event_type": "view", "event_ts": "2026-10-02 09:00:00"}',
        '{"event_id": "e5", "customer_id": "c4", "event_type": "cart", "event_ts": "2026-10-03 10:00:00", "items": [{"sku": "C", "qty": 3}]',
        '{"event_id": "e6", "customer_id": "c4", "event_type": "cart", "event_ts": "2026-10-03 10:05:00", "items": [{"sku": "C", "qty": 3}]}',
    ],
}


def land(name: str, rows: list[str]) -> str:
    """Write one raw extract into the landing folder and return its path."""
    path = os.path.join(LANDING, name)
    with open(path, "w", encoding="utf-8") as handle:
        handle.write("\n".join(rows) + "\n")
    return path


landed = {name: land(name, rows) for name, rows in RAW_FILES.items()}
print("landed:", sorted(os.listdir(LANDING)))

## Stage 1 - Bronze: typed reads and a quarantine

Uses: explicit schemas and read modes ([PySpark 2](pyspark-02-reading-sql-and-cleaning.ipynb)) and bad-record handling ([PySpark 5](pyspark-05-writing-pipelines-and-quality.ipynb)).

The schemas are declared up front, never inferred. `PERMISSIVE` mode with a
`_corrupt_record` column keeps unparseable rows instead of failing the job or silently dropping
them. Rows that parse but break a business rule (no key, negative amount) are flagged too. Both
kinds go to a quarantine table with a reason, and only clean rows move on. Each row also carries
its source file name (from the `_metadata` column) and a batch id, which the dedupe and the
re-run rely on later.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    ArrayType,
    DoubleType,
    IntegerType,
    LongType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)

TS_FORMAT = "yyyy-MM-dd HH:mm:ss"
CORRUPT = "_corrupt_record"

ORDERS_SCHEMA = StructType([
    StructField("order_id", LongType()),
    StructField("customer_id", StringType()),
    StructField("amount", DoubleType()),
    StructField("status", StringType()),
    StructField("order_ts", TimestampType()),
    StructField("updated_at", TimestampType()),
    StructField(CORRUPT, StringType()),
])

CUSTOMERS_SCHEMA = StructType([
    StructField("customer_id", StringType()),
    StructField("name", StringType()),
    StructField("country", StringType()),
])

ITEM = StructType([StructField("sku", StringType()), StructField("qty", IntegerType())])
EVENTS_SCHEMA = StructType([
    StructField("event_id", StringType()),
    StructField("customer_id", StringType()),
    StructField("event_type", StringType()),
    StructField("event_ts", TimestampType()),
    StructField("items", ArrayType(ITEM)),
    StructField(CORRUPT, StringType()),
])

### Read orders into bronze

The DataFrame is cached before it is filtered: Spark refuses a query on a raw CSV or JSON file
that references only the corrupt-record column, and caching also stops the file from being
parsed twice for the valid and rejected branches.

In [ ]:
def read_bronze_orders(paths: list[str], batch_id: str):
    """Return (valid, rejected) bronze orders for one batch of landed CSV files."""
    raw = (
        spark.read.schema(ORDERS_SCHEMA)
        .option("header", True)
        .option("mode", "PERMISSIVE")
        .option("columnNameOfCorruptRecord", CORRUPT)
        .option("timestampFormat", TS_FORMAT)
        .csv(paths)
        .withColumn("source_file", F.col("_metadata.file_name"))
        .withColumn("batch_id", F.lit(batch_id))
        .cache()
    )
    reason = (
        F.when(F.col(CORRUPT).isNotNull(), "unparseable row")
        .when(F.col("order_id").isNull(), "missing order_id")
        .when(F.col("amount") < 0, "negative amount")
        .when(F.col("amount").isNull(), "missing amount")
        .when(F.col("customer_id").isNull(), "missing customer_id")
        .when(F.col("order_ts").isNull() | F.col("updated_at").isNull(), "missing timestamp")
        .when(F.length(F.trim(F.col("status"))).isNull() | (F.trim(F.col("status")) == ""), "missing status")
    )
    flagged = raw.withColumn("reject_reason", reason)
    valid = flagged.filter(F.col("reject_reason").isNull()).drop(CORRUPT, "reject_reason")
    rejected = flagged.filter(F.col("reject_reason").isNotNull()).select(
        "batch_id", "source_file", "reject_reason", F.col(CORRUPT).alias("raw_line"), "order_id"
    )
    return valid, rejected


bronze_orders, rejected_orders = read_bronze_orders([landed["orders_day1.csv"]], "day1")
rejected_orders.show(truncate=False)
print("valid bronze orders:", bronze_orders.count())

### Persist the quarantine

The quarantine is partitioned by `batch_id` and written with an overwrite. With dynamic
partition overwrite that replaces only this batch's partition, so re-running a batch never
doubles its rejects.

In [ ]:
QUARANTINE = os.path.join(LAKE, "quarantine", "orders")


def write_quarantine(rejected) -> None:
    """Replace this batch's partition of the quarantine table."""
    rejected.write.mode("overwrite").partitionBy("batch_id").parquet(QUARANTINE)


write_quarantine(rejected_orders)
reasons = sorted(r["reject_reason"] for r in spark.read.parquet(QUARANTINE).collect())
print("quarantined:", reasons)

In [ ]:
assert bronze_orders.count() == 7, bronze_orders.count()
assert reasons == ["missing order_id", "negative amount", "unparseable row", "unparseable row"], reasons
assert bronze_orders.count() + len(reasons) == len(RAW_FILES["orders_day1.csv"]) - 1  # every row accounted for
assert "source_file" in bronze_orders.columns and CORRUPT not in bronze_orders.columns
print("stage 1 passed: 7 valid rows, 4 quarantined with reasons, nothing lost")

## Stage 2 - Silver: clean, deduplicate, flatten

Uses: string cleanup ([PySpark 2](pyspark-02-reading-sql-and-cleaning.ipynb)), `row_number` over a window and `explode_outer` on arrays ([PySpark 4](pyspark-04-windows-and-nested-data.ipynb)).

Orders keep the **latest version** of each `order_id` by `updated_at`, with `source_file` as a
tie-breaker so the winner is deterministic even for exact duplicates. Customers are trimmed and
upper-cased. Events are deduplicated on `event_id`, and their `items` array is exploded with
`explode_outer` so events with an empty or missing array are not silently dropped.

In [ ]:
from pyspark.sql.window import Window

LATEST_FIRST = Window.partitionBy("order_id").orderBy(
    F.col("updated_at").desc(), F.col("source_file").desc()
)


def standardize_orders(df):
    """Normalise status text and derive the partition column."""
    return df.withColumn("status", F.lower(F.trim("status"))).withColumn("order_date", F.to_date("order_ts"))


def latest_per_order(df):
    """Keep one row per order_id: the most recent update."""
    return df.withColumn("rn", F.row_number().over(LATEST_FIRST)).filter(F.col("rn") == 1).drop("rn")


def read_silver_customers(path: str):
    """Trimmed names and upper-case ISO country codes."""
    return (
        spark.read.schema(CUSTOMERS_SCHEMA).option("header", True).csv(path)
        .withColumn("name", F.trim("name"))
        .withColumn("country", F.upper(F.trim("country")))
    )


def read_silver_events(paths: list[str]):
    """Return (events, event_items): deduplicated events and one row per item."""
    raw = (
        spark.read.schema(EVENTS_SCHEMA)
        .option("mode", "PERMISSIVE")
        .option("columnNameOfCorruptRecord", CORRUPT)
        .option("timestampFormat", TS_FORMAT)
        .json(paths)
        .cache()
    )
    events = (
        raw.filter(F.col(CORRUPT).isNull() & F.col("event_id").isNotNull())
        .drop(CORRUPT)
        .dropDuplicates(["event_id"])
    )
    event_items = events.select(
        "event_id", "customer_id", "event_type", F.explode_outer("items").alias("item")
    ).select("event_id", "customer_id", "event_type", "item.sku", "item.qty")
    return events, event_items


silver_day1 = latest_per_order(standardize_orders(bronze_orders))
silver_customers = read_silver_customers(landed["customers.csv"])
silver_events, silver_event_items = read_silver_events([landed["events_day1.jsonl"]])

silver_day1.orderBy("order_id").select("order_id", "customer_id", "amount", "status", "order_date").show()
silver_customers.show()

In [ ]:
by_id = {r["order_id"]: r for r in silver_day1.collect()}
assert sorted(by_id) == [1, 2, 3, 4, 5]
assert by_id[3]["status"] == "delivered"  # the later update won
assert by_id[2]["status"] == "paid"  # "PAID " was normalised
assert [r["country"] for r in silver_customers.orderBy("customer_id").collect()] == ["US", "GB", "FI", "US"]
assert silver_events.count() == 5  # e4 deduplicated, truncated e5 dropped
assert silver_event_items.count() == 6  # e3 and e4 kept as one null-item row each
assert silver_events.select(F.explode("items")).count() == 4  # plain explode would lose them
print("stage 2 passed: latest-row dedupe, clean dimension, explode_outer keeps empty arrays")

## Stage 3 - Write silver, build gold

Uses: partitioned Parquet writes ([PySpark 5](pyspark-05-writing-pipelines-and-quality.ipynb)), joins and `groupBy`/`pivot` ([PySpark 3](pyspark-03-joins-and-aggregations.ipynb)), and ranking
windows ([PySpark 4](pyspark-04-windows-and-nested-data.ipynb)).

Silver orders are written partitioned by `order_date`. The write goes through a staging folder
first because Spark refuses to overwrite a path in the same job that reads from it, which the
day-2 upsert in Stage 4 will do. `repartition("order_date")` before the final write gives one
file per date instead of one per task.

Gold is rebuilt from silver on every run. Revenue counts only `paid`, `shipped` and `delivered`
orders, and the customer dimension is joined with an explicit `broadcast` hint and a `left` join,
so an order for an unknown customer lands under `UNKNOWN` instead of disappearing.

In [ ]:
SILVER_ORDERS = os.path.join(LAKE, "silver", "orders")
STAGING_ORDERS = os.path.join(LAKE, "_staging", "orders")
REVENUE_STATUSES = ["paid", "shipped", "delivered"]


def upsert_silver_orders(batch_orders) -> list:
    """Merge a batch into silver, rewriting only the order_date partitions it touches."""
    affected = sorted(r["order_date"] for r in batch_orders.select("order_date").distinct().collect())
    if not affected:
        return []
    merged = batch_orders
    if os.path.exists(SILVER_ORDERS):
        existing = spark.read.parquet(SILVER_ORDERS).filter(F.col("order_date").isin(affected))
        merged = existing.unionByName(batch_orders)
    latest_per_order(merged).write.mode("overwrite").parquet(STAGING_ORDERS)
    (
        spark.read.parquet(STAGING_ORDERS)
        .repartition("order_date")
        .write.mode("overwrite")
        .partitionBy("order_date")
        .parquet(SILVER_ORDERS)
    )
    return affected


def build_gold(orders, customers, events, event_items):
    """Return (revenue_orders, daily_revenue, customer_summary) from silver inputs."""
    revenue_orders = (
        orders.filter(F.col("status").isin(REVENUE_STATUSES))
        .join(F.broadcast(customers.select("customer_id", "country")), "customer_id", "left")
        .withColumn("country", F.coalesce("country", F.lit("UNKNOWN")))
    )
    daily_revenue = revenue_orders.groupBy("order_date", "country").agg(
        F.round(F.sum("amount"), 2).alias("revenue"), F.count("*").alias("orders")
    )
    lifetime = revenue_orders.groupBy("customer_id").agg(
        F.round(F.sum("amount"), 2).alias("lifetime_revenue"), F.count("*").alias("paid_orders")
    )
    activity = events.groupBy("customer_id").pivot("event_type", ["view", "cart"]).count()
    units = (
        event_items.filter(F.col("event_type") == "cart")
        .groupBy("customer_id")
        .agg(F.sum("qty").alias("units_carted"))
    )
    by_country = Window.partitionBy("country").orderBy(F.desc("lifetime_revenue"))
    customer_summary = (
        customers.join(lifetime, "customer_id", "left")
        .join(activity, "customer_id", "left")
        .join(units, "customer_id", "left")
        .na.fill(0, ["lifetime_revenue", "paid_orders", "view", "cart", "units_carted"])
        .withColumn("country_rank", F.dense_rank().over(by_country))
    )
    return revenue_orders, daily_revenue, customer_summary


affected_day1 = upsert_silver_orders(silver_day1)
orders_silver = spark.read.parquet(SILVER_ORDERS)
revenue_orders, daily_revenue, customer_summary = build_gold(
    orders_silver, silver_customers, silver_events, silver_event_items
)
daily_revenue.orderBy("order_date", "country").show()
customer_summary.orderBy("customer_id").show()

In [ ]:
daily = {(str(r["order_date"]), r["country"]): r["revenue"] for r in daily_revenue.collect()}
summary = {r["customer_id"]: r for r in customer_summary.collect()}
assert [str(d) for d in affected_day1] == ["2026-10-01", "2026-10-02", "2026-10-03"]
assert daily == {
    ("2026-10-01", "US"): 120.5,
    ("2026-10-01", "GB"): 80.0,
    ("2026-10-02", "FI"): 200.0,
    ("2026-10-03", "UNKNOWN"): 60.0,  # c9 is not in the dimension but its revenue is kept
}
assert summary["c1"]["paid_orders"] == 1  # the cancelled order is not revenue
assert (summary["c1"]["country_rank"], summary["c4"]["country_rank"]) == (1, 2)
assert (summary["c1"]["units_carted"], summary["c4"]["units_carted"]) == (3, 3)
print("stage 3 passed: silver partitioned by date, gold revenue and customer summary correct")

## Stage 4 - Incremental re-run on day 2

Uses: incremental loads and idempotent writes ([PySpark 5](pyspark-05-writing-pipelines-and-quality.ipynb)).

Day 2 brings new orders, a late refund for order 5 (a day-1 date), a **stale** copy of order 3
whose `updated_at` is older than the version silver already holds, and one corrupt row. The
whole pipeline is wrapped in one function and run on it **twice**. A correct incremental job
must:

1. rewrite only the `order_date` partitions the batch touches, leaving `2026-10-01` alone;
2. let the newest `updated_at` win, so the stale order 3 does not regress `delivered` to `shipped`;
3. produce identical silver, gold and quarantine output when the same batch is re-run.

Nothing in the output depends on the wall clock (no `current_timestamp()`), which is what makes
point 3 possible.

In [ ]:
landed["orders_day2.csv"] = land(
    "orders_day2.csv",
    [
        ORDERS_HEADER,
        "5,c9,60.00,refunded,2026-10-03 08:45:00,2026-10-04 09:00:00",
        "3,c3,200.00,shipped,2026-10-02 09:15:00,2026-10-02 09:15:00",
        "9,c4,99.00,paid,2026-10-04 10:00:00,2026-10-04 10:00:00",
        "10,c2,20.00,paid,2026-10-04 11:00:00,2026-10-04 11:00:00",
        "11,c3,12.x,paid,2026-10-04 12:00:00,2026-10-04 12:00:00",
    ],
)


def run_pipeline(order_files: list[str], batch_id: str) -> dict:
    """Run bronze -> silver -> gold for one batch of order files and write every layer."""
    valid, rejected = read_bronze_orders(order_files, batch_id)
    write_quarantine(rejected)
    affected = upsert_silver_orders(standardize_orders(valid))
    _, daily, summary = build_gold(
        spark.read.parquet(SILVER_ORDERS), silver_customers, silver_events, silver_event_items
    )
    daily.write.mode("overwrite").parquet(os.path.join(LAKE, "gold", "daily_revenue"))
    summary.write.mode("overwrite").parquet(os.path.join(LAKE, "gold", "customer_summary"))
    return {"affected": [str(d) for d in affected], "rejected": rejected.count()}


def snapshot(path: str) -> list[tuple]:
    """Sorted rows of a table, for exact before/after comparison."""
    return sorted(tuple(str(v) for v in row) for row in spark.read.parquet(path).collect())


def files_in(partition: str) -> list[str]:
    """Data file names inside one silver partition folder."""
    folder = os.path.join(SILVER_ORDERS, partition)
    return sorted(f for f in os.listdir(folder) if f.endswith(".parquet"))


untouched_before = files_in("order_date=2026-10-01")
first = run_pipeline([landed["orders_day2.csv"]], "day2")
silver_after_first = snapshot(SILVER_ORDERS)
gold_after_first = snapshot(os.path.join(LAKE, "gold", "daily_revenue"))

second = run_pipeline([landed["orders_day2.csv"]], "day2")  # the scheduler retries the same batch
print("first run :", first)
print("second run:", second)
spark.read.parquet(SILVER_ORDERS).orderBy("order_id").select("order_id", "status", "order_date", "batch_id").show()

In [ ]:
silver_now = {r["order_id"]: r for r in spark.read.parquet(SILVER_ORDERS).collect()}
quarantine_batches = [r["batch_id"] for r in spark.read.parquet(QUARANTINE).collect()]
assert first == second == {"affected": ["2026-10-02", "2026-10-03", "2026-10-04"], "rejected": 1}
assert files_in("order_date=2026-10-01") == untouched_before  # partition not rewritten
assert silver_now[3]["status"] == "delivered"  # the stale update did not win
assert silver_now[5]["status"] == "refunded"  # the late update did
assert sorted(silver_now) == [1, 2, 3, 4, 5, 9, 10]
assert snapshot(SILVER_ORDERS) == silver_after_first  # re-run changed nothing
assert snapshot(os.path.join(LAKE, "gold", "daily_revenue")) == gold_after_first
assert sorted(quarantine_batches) == ["day1"] * 4 + ["day2"]  # rejects not doubled
empty_batch = standardize_orders(bronze_orders).limit(0)
assert upsert_silver_orders(empty_batch) == []
assert snapshot(SILVER_ORDERS) == silver_after_first
print("stage 4 passed: only touched partitions rewritten, latest update wins, re-run is idempotent")

## Stage 5 - Audit the resulting tables

Inspect the published sample tables after both deliveries. Check uniqueness, missing keys and amounts, and allowed statuses. The missing customer `c9` is a known warning. Its order stays in the data and appears under `UNKNOWN` when eligible for revenue.

Compare eligible silver revenue with gold revenue. A production job should run its publication checks on the staged result before switching readers to the new version. This local walkthrough audits files already written in the preceding step.

In [ ]:
KNOWN_STATUSES = ["paid", "shipped", "delivered", "cancelled", "refunded"]


def quality_report(orders, customers) -> dict:
    """Count violations of each rule; zero means the rule holds."""
    return {
        "duplicate_order_ids": orders.groupBy("order_id").count().filter(F.col("count") > 1).count(),
        "null_keys": orders.filter(F.col("order_id").isNull() | F.col("customer_id").isNull()).count(),
        "negative_amounts": orders.filter(F.col("amount") < 0).count(),
        "unknown_statuses": orders.filter(F.col("status").isNull() | ~F.col("status").isin(KNOWN_STATUSES)).count(),
        "orphan_customers": orders.join(customers, "customer_id", "left_anti").count(),
    }


orders_silver = spark.read.parquet(SILVER_ORDERS)
report = quality_report(orders_silver, silver_customers)
print(report)

silver_revenue = orders_silver.filter(F.col("status").isin(REVENUE_STATUSES)).agg(F.sum("amount")).first()[0]
gold_revenue = spark.read.parquet(os.path.join(LAKE, "gold", "daily_revenue")).agg(F.sum("revenue")).first()[0]
print("silver revenue:", silver_revenue, "| gold revenue:", gold_revenue)

In [ ]:
hard_rules = {k: v for k, v in report.items() if k != "orphan_customers"}
assert all(v == 0 for v in hard_rules.values()), hard_rules
assert report["orphan_customers"] == 1, report  # c9: known, reported under UNKNOWN
assert round(silver_revenue, 2) == round(gold_revenue, 2) == 519.5
print("stage 5 passed: hard rules hold, one known orphan, gold reconciles to silver")

## Stage 6 - A quick performance check

Uses: reading physical plans, broadcast tuning, partition pruning and file sizing ([PySpark 7](pyspark-07-monitoring-and-performance.ipynb)).

Before shipping, read the plan instead of guessing. Three checks that catch most regressions in
a pipeline like this: the dimension join is a `BroadcastHashJoin` (and falls back to a
`SortMergeJoin` with a shuffle when broadcasting is off), a date filter on silver prunes
partitions instead of scanning them all, and each partition holds one file rather than many
small ones.

In [ ]:
import re


def physical_plan(df) -> str:
    """The physical plan Spark will run, as text."""
    return df._jdf.queryExecution().executedPlan().toString()


revenue_orders, _, _ = build_gold(orders_silver, silver_customers, silver_events, silver_event_items)
hinted_plan = physical_plan(revenue_orders)

spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
no_broadcast_plan = physical_plan(orders_silver.join(silver_customers, "customer_id", "left"))
spark.conf.unset("spark.sql.autoBroadcastJoinThreshold")

one_day = orders_silver.filter(F.col("order_date") == F.lit("2026-10-04").cast("date"))
scan_plan = physical_plan(one_day)
partition_filters = re.search(r"PartitionFilters: \[([^\]]*)\]", scan_plan).group(1)

files_per_partition = {
    folder: len(files_in(folder)) for folder in sorted(os.listdir(SILVER_ORDERS)) if folder.startswith("order_date=")
}
print("hinted join       :", "BroadcastHashJoin" if "BroadcastHashJoin" in hinted_plan else "other")
print("broadcast off     :", "SortMergeJoin" if "SortMergeJoin" in no_broadcast_plan else "other")
print("partition filters :", partition_filters)
print("files per date    :", files_per_partition)

In [ ]:
assert hinted_plan.startswith("AdaptiveSparkPlan"), hinted_plan[:80]  # AQE is in charge
assert "BroadcastHashJoin" in hinted_plan, hinted_plan
assert "SortMergeJoin" in no_broadcast_plan and "Exchange" in no_broadcast_plan, no_broadcast_plan
assert "order_date" in partition_filters, scan_plan
assert one_day.count() == 2
assert set(files_per_partition.values()) == {1}, files_per_partition
print("stage 6 passed: broadcast join, partition pruning, one file per partition")

### Running this on a platform

The logic carries over unchanged; what changes is the storage layer and the scheduler.

| Concern | This notebook | Databricks / Delta Lake | Snowflake |
| --- | --- | --- | --- |
| Landing to bronze | `spark.read` on a folder | Auto Loader with a schema and rescued-data column | `COPY INTO` with `ON_ERROR` and a stage |
| Idempotent upsert | Dynamic partition overwrite | `MERGE INTO` keyed on `order_id` | `MERGE` in a task or dynamic table |
| Small files | `repartition` before write | `OPTIMIZE` and auto-compaction | Managed by the engine |
| Quality gate | `assert` on a report dict | Expectations in a declarative pipeline | Data metric functions or a test task |

## Reconcile the pipeline before publishing

**Reconciliation** accounts for what happened to every input record. At the bronze boundary, accepted rows plus rejected rows should equal landed rows if the categories are disjoint. At the silver boundary, explain the difference between accepted rows and output rows with counts for duplicate or superseded records. At gold, compare totals at the same business grain and filter scope.

**Where it matters.** A backfill can finish successfully and still omit a file or multiply records at a join. Counts and totals give the operator evidence before the published table changes.

**Watch out for.** A count alone misses offsetting errors. Check keys and amounts together. Track business dates separately from delivery dates so late records are reconciled against the reporting slice they affect.

### Example

A delivery has 100 raw rows. Bronze accepts 98 and quarantines 2. Silver drops 3 retries, leaving 95 unique orders. Gold should account for those 95 orders, and its revenue should equal the sum of amounts for the same 95 silver orders. If a check fails, keep the prior published result and investigate the affected stage.

## Interview questions

Twelve questions this pipeline lets you answer from experience. Say the answer out loud first,
then point to the stage above that demonstrates it. Each answer links to the module that goes
deeper.

**1. Walk me through a bronze, silver, gold pipeline you have built.**

Bronze is the raw data typed with an explicit schema plus lineage columns (source file, batch),
with bad rows quarantined rather than dropped. Silver is cleaned, deduplicated and conformed to
one row per business key. Gold is shaped for consumers: aggregates and joined summaries. Each
layer is rebuildable from the one before it, so a bug fix is a re-run, not a manual patch.
See [PySpark 5](pyspark-05-writing-pipelines-and-quality.ipynb) and [PySpark 8](pyspark-08-real-world-and-interviews.ipynb).

**2. How do you handle malformed records when reading files?**

Choose the read mode on purpose: `FAILFAST` when any bad row should stop the job, `DROPMALFORMED`
almost never (it hides data loss), and usually `PERMISSIVE` with `columnNameOfCorruptRecord`, so
bad rows are kept, routed to a quarantine with a reason, and counted. Reconcile input rows to
valid plus rejected so nothing disappears. See [PySpark 2](pyspark-02-reading-sql-and-cleaning.ipynb) and [PySpark 5](pyspark-05-writing-pipelines-and-quality.ipynb).

**3. How do you make a batch job idempotent?**

A re-run of the same input must leave the same output. Overwrite exactly the slice the batch owns
(dynamic partition overwrite or a `MERGE` on the key) instead of appending, make dedupe
deterministic with a tie-breaker, and keep wall-clock values such as `current_timestamp()` out of
the data. Test it by running the batch twice and comparing snapshots, as Stage 4 does. See [PySpark 5](pyspark-05-writing-pipelines-and-quality.ipynb).

**4. How do you keep the latest record per key?**

`row_number()` over `Window.partitionBy(key).orderBy(updated_at desc, tie_breaker desc)` and keep
`rn = 1`. Without a tie-breaker the winner among equal timestamps can change between runs.
`dropDuplicates` is fine for exact duplicates but gives no control over which row survives.
See [PySpark 4](pyspark-04-windows-and-nested-data.ipynb).

**5. What is a shuffle, and why is it expensive?**

A shuffle redistributes rows across partitions so rows sharing a key land together. It writes
intermediate data to local disk, sends it over the network, reads it back, can spill, and forces
a stage boundary. Wide operations cause it (`groupBy`, `join`, `distinct`, `repartition`, windows
with a partition key); narrow ones (`filter`, `select`, `withColumn`) do not. Every `Exchange` in
`explain()` is one. See [PySpark 7](pyspark-07-monitoring-and-performance.ipynb).

**6. When does a broadcast join help, and what are its limits?**

When one side is small enough to copy to every executor: the large side is joined in place with
no shuffle. The limit is executor and driver memory, not the 10 MB default of
`spark.sql.autoBroadcastJoinThreshold`; a hint on a table that is not small causes OOMs. It does
not fix skew on the large side. Confirm it in the plan (`BroadcastHashJoin`), as Stage 6 does.
See [PySpark 3](pyspark-03-joins-and-aggregations.ipynb) and [PySpark 7](pyspark-07-monitoring-and-performance.ipynb).

**7. How do you handle data skew, and what can AQE not fix?**

Spot it as one task far slower than its peers with much more shuffle read. Let AQE split skewed
join partitions first; then broadcast the small side, salt the hot key, or fix the root cause
(often a null or placeholder key). AQE works on shuffle statistics, so it cannot fix a skewed
source read or a single hot key in an aggregation. See [PySpark 7](pyspark-07-monitoring-and-performance.ipynb).

**8. How do you choose partition columns and avoid small files?**

Partition on a low-cardinality column that almost every query filters on (usually a date), aiming
for partitions of hundreds of MB. Over-partitioning creates many tiny files that slow listing and
planning. Control file count at write time with `repartition(partition_col)` or `coalesce`, and
compact on a schedule. `repartition` shuffles to balance; `coalesce` only merges, without a
shuffle. See [PySpark 7](pyspark-07-monitoring-and-performance.ipynb).

**9. When should you cache, and how is that different from checkpointing?**

Cache when a DataFrame is reused by several actions, as bronze is here for its valid and rejected
branches; `unpersist` when done, and do not cache something used once. Caching keeps the lineage
in memory or local disk for the session; checkpointing writes to reliable storage and truncates
the lineage, which suits very long plans and streaming state. See [PySpark 7](pyspark-07-monitoring-and-performance.ipynb).

**10. Why are Python UDFs slow, and what do you use instead?**

Rows are serialised between the JVM and a Python worker, processed one at a time, and hidden
from the optimizer. Use built-in functions first (this whole pipeline needs none), then
higher-order array functions, then a vectorised pandas UDF. Keep plain Python UDFs for cold paths.
See [PySpark 5](pyspark-05-writing-pipelines-and-quality.ipynb) and [PySpark 7](pyspark-07-monitoring-and-performance.ipynb).

**11. How would this pipeline change as a stream, and how is exactly-once achieved?**

The layers stay; the reads become `readStream` and the writes `writeStream` with a checkpoint.
Exactly-once needs a replayable source with offsets, a durable checkpoint of offsets and state,
and an idempotent or transactional sink (Delta, or a `foreachBatch` `MERGE`). Late data needs a
watermark to bound state. See [PySpark 6](pyspark-06-structured-streaming.ipynb).

**12. How do you debug a slow Spark job?**

Find the longest stage in the Spark UI, then the slowest task in it, and compare its input and
spill to its peers to spot skew. Read `explain()` for unexpected `Exchange` nodes, sort-merge
joins that should broadcast, and scans with no partition filters. Check partition counts, spill
and GC time, and whether a UDF or `collect()` sits on the hot path. Change one thing and measure
again. See [PySpark 7](pyspark-07-monitoring-and-performance.ipynb).

### Next steps

* [SQL 8: window functions](../sql/sql-08-window-functions.ipynb) - the same window logic in Spark SQL.
* [PySpark coding challenges](pyspark-coding-challenges.ipynb) - interview-style problems with hidden solutions.
* [Interview question bank](../interview-questions/interview-question-bank.ipynb) - a mixed question bank for mock interviews.

In [ ]:
import shutil

spark.stop()
shutil.rmtree(LAKE, ignore_errors=True)
print("SparkSession stopped and temporary lake removed.")